In [1]:
import ast
from beamngpy import BeamNGpy, Scenario, Vehicle
from pathlib import Path
import time
from shapely.geometry import Polygon, Point

In [5]:
# Load track layout and settings
with open('Data/train_map.txt') as file:
    train_map = file.read()

with open('Data/train_SETTINGS.txt') as file:
    train_settings = file.read()

In [6]:
# Convert String to dictionary/list
train_map = ast.literal_eval(train_map)
train_settings = ast.literal_eval(train_settings)

In [7]:
PATH = Path("../../BeamNG")
POS = train_settings['START_POS_MID']
ROT_QUAT = train_settings['ROT_QUAT']

In [ ]:
class BNG:
    def __init__(self, PATH, POS, ROT_QUAT, train_map, train_settings):
        self.PATH = PATH
        self.POS = POS
        self.ROT_QUAT = ROT_QUAT
        self.train_map = train_map
        self.train_settings = train_settings
        self.build_track_polygon()

    def initialize(self):
        # Open the game
        self.bng = BeamNGpy("localhost", 25252, self.PATH)
        self.bng.open()

        # Create the scenario
        scenario = Scenario("automation_test_track", "Training_Cycle")
        self.v = Vehicle('agent', model="etk800", part_config='vehicles/etk800/846x_ttsport_plus_DCT.pc', color="0.878 0.447 0.0 1.0", license="AG3NT")
        scenario.add_vehicle(self.v, pos=self.POS, rot_quat=self.ROT_QUAT)
        scenario.make(self.bng)

        # Load and start the scenario
        self.bng.scenario.load(scenario)
        self.bng.scenario.start()

    def build_track_polygon(self):
        left_edge = [edge['left'][:2] for edge in self.train_map]
        right_edge = [edge['right'][:2] for edge in self.train_map]
        polygon = left_edge + right_edge[::-1]
        self.poly = Polygon(polygon)
        
    def check_pos(self, margin=0.5):
        self.v.sensors.poll()
        point = Point(self.v.state["pos"][:2])
        print(point.distance(self.poly))
        return self.poly.covers(point) or point.distance(self.poly) <= margin

    def reset(self):
        # self.v.recover()
        self.v.teleport(pos=self.POS, rot_quat=self.ROT_QUAT)

    def close(self):
        self.bng.close()

In [ ]:
bng = BNG(PATH, POS, ROT_QUAT, train_map, train_settings)
bng.initialize()
i = 0
while bng.check_pos():
    print(i)
    i += 1
bng.reset()
time.sleep(10)
bng.close()
